# Kata 04 — Aislamiento de Subagentes (Hub-and-Spoke)

> Spec: [`specs/004-subagent-isolation/spec.md`](../../specs/004-subagent-isolation/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
from katas._shared.eventlog import Logger
import json

client, settings = bootstrap(budget_calls=15)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Un coordinador descompone una tarea grande, lanza N subagentes en paralelo (cada uno una llamada `client.messages.create` independiente con prompt enfocado) y agrega los resultados tipados. Cero memoria compartida, cero historial heredado entre el coordinador y los subagentes.

## 2. Por qué importa

Pasar todo el historial del coordinador al subagente "para que tenga contexto" diluye su atención (Kata 11), filtra políticas que el subagente no debería ver, y multiplica el costo por la cantidad de subagentes. Cada subagente es una sesión nueva.

## 3. Ejemplo correcto — fan-out con prompts focalizados

In [ ]:
DOCS = {
    "doc-A": "ARR Q3 2025 = 12M USD. Headcount al cierre del trimestre: 450 personas.",
    "doc-B": "Ingresos del año fiscal 2025: $48M. Equipo: 462 colaboradores en diciembre 2025.",
    "doc-C": "El churn anual en 2025 fue 8%. ARR de cierre: 12.4M USD.",
}

EXTRACT_TOOL = {
    "name": "submit_findings",
    "description": "Devuelve los hallazgos del documento en formato tipado.",
    "input_schema": {
        "type": "object",
        "properties": {
            "facts": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "topic": {"type": "string", "enum": ["arr", "headcount", "churn", "other"]},
                        "value": {"type": "string"},
                    },
                    "required": ["topic", "value"],
                },
            }
        },
        "required": ["facts"],
    },
}

def run_subagent(client, doc_id: str, doc_text: str) -> dict:
    """Sesión nueva, prompt mínimo, salida tipada."""
    system = "Extrae hechos clave del documento. Llama a submit_findings con la lista."
    resp = client.messages.create(
        system=system,
        messages=[{"role": "user", "content": f"DOC ID: {doc_id}\n\n{doc_text}"}],
        tools=[EXTRACT_TOOL],
        tool_choice={"type": "tool", "name": "submit_findings"},
    )
    tu = next(b for b in resp.content if b.type == "tool_use")
    return {"doc_id": doc_id, "facts": tu.input["facts"]}

In [ ]:
results = [run_subagent(client, doc_id, text) for doc_id, text in DOCS.items()]
for r in results:
    print(r["doc_id"], "→", r["facts"])

### 3.1 Coordinador agrega lo tipado, no lo crudo

In [ ]:
def coordinator_aggregate(results):
    by_topic = {}
    for r in results:
        for f in r["facts"]:
            by_topic.setdefault(f["topic"], []).append({"source": r["doc_id"], "value": f["value"]})
    return by_topic

print(json.dumps(coordinator_aggregate(results), indent=2, ensure_ascii=False))

Cada subagente vio sólo su documento. El coordinador nunca ejecutó `messages.create` con `DOC_A + DOC_B + DOC_C` concatenados ni con history compartida.

## 4. Anti-patrón — un único agente con todo el contexto

In [ ]:
def single_agent_dump(client):
    big_prompt = "\n\n".join(f"DOC {k}:\n{v}" for k, v in DOCS.items())
    system = "Eres un analista. Para cada doc, extrae hechos clave."
    resp = client.messages.create(
        system=system,
        messages=[{"role": "user", "content": big_prompt}],
    )
    return resp

bad = single_agent_dump(client)
print("texto:", next((b.text for b in bad.content if b.type == "text"), "")[:400])
print("input_tokens:", bad.usage.input_tokens)

Diferencias observables:

- **Tipado vs prosa**: el subagente correcto devuelve JSON tipado (verificable, agregable, auditable). El monolítico devuelve prosa libre, propensa a omitir docs y mezclar hechos.
- **Aislamiento**: si DOC_C contiene un prompt injection, sólo afecta al subagente de DOC_C. En el monolítico, contamina todo.
- **Costo**: el monolítico paga por todos los docs en cada turno. El fan-out paga 1 doc por subagente.

## 5. Argumento de certificación

- **Hub-and-spoke estricto**: una llamada nueva al SDK por subagente, prompt mínimo, salida tipada por schema (`tool_choice` forzado).
- **Sin telepatía compartida**: pasar `coordinator.history` al subagente está prohibido por Principio IV.
- **Aislamiento como defensa**: prompt injection en un doc no escala; el blast radius es un solo subagente.
- **Conexión con otros katas**: cada subagente puede ser un Kata 12 (chaining) interno; los resultados tipados se agregan respetando provenance (Kata 20).

## 6. Auto-evaluación

**1. ¿Cómo demuestro que el subagente NO recibió historial del coordinador?**

`run_subagent` arma `messages=[{"role":"user", ...}]` desde cero por cada llamada. No hay variable `coordinator_history` que se pase. Si quisiera evidencia adicional: imprimir `len(messages)` antes de cada `create` — siempre debe ser 1.

**2. ¿Qué hago si dos subagentes producen JSONs en conflicto?**

Los registro ambos en el agregado (ver `coordinator_aggregate`: cada `value` viene con `source`). La resolución se delega: si el conflicto importa, escala a humano (Kata 16); si no, queda como dato preservado con provenance (Kata 20). Nunca elijo uno arbitrariamente.

**3. ¿Cuándo está justificado romper el aislamiento?**

Casi nunca. La excepción razonable es pasar artefactos compartidos read-only (un schema, una guía de estilo) en el system prompt — pero eso es contexto estable, no historial conversacional. Si me veo tentado a pasar history, primero pregunto si el problema se resuelve con un mejor prompt o partiendo distinto la tarea.